In [6]:
!pip install curl_cffi beautifulsoup4 pandas

In [8]:
import os
import time
import random
import re
import json
import urllib.parse
import pandas as pd
from bs4 import BeautifulSoup
from curl_cffi import requests as curl_requests
import sys
import warnings

warnings.simplefilter(action='ignore', category=FutureWarning)

print("Starting the Autonomous TTRPG Feature Engineering Pipeline (HTML SCRAPE + curl_cffi)...")

# --- SETUP ---
INPUT_FILE = "ttrpg_database_final.csv"
SCRAPED_FILE = "scraped_rpg_features_part3.csv"
FINAL_OHE_FILE = "ttrpg_features_ohe_final.csv"

# ==========================================
# BLOCK 1: Stratified Sampling (Exactly 500 per Tier)
# ==========================================
print("\n--- BLOCK 1: STRATIFIED SAMPLING DATA ---")
try:
    df_main = pd.read_csv(INPUT_FILE)
    print(f"Loaded main dataset with {len(df_main)} total games.")
except FileNotFoundError:
    print(f"Error: Could not find '{INPUT_FILE}'.")
    sys.exit()

if os.path.exists(SCRAPED_FILE):
    df_sampled = pd.read_csv(SCRAPED_FILE)
    print(f"Resuming from existing '{SCRAPED_FILE}' with {len(df_sampled)} games.")
else:
    bins = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
    labels = ['0-1', '1-2', '2-3', '3-4', '4-5', '5-6', '6-7', '7-8', '8-9', '9-10']
    df_main['Score_Tier'] = pd.cut(df_main['Average Score'], bins=bins, labels=labels)
    print("Bucketing games by score and sampling exactly 500 per tier...")
    df_sampled = df_main.groupby('Score_Tier', group_keys=False, observed=False).apply(
        lambda x: x.sample(n=500, replace=len(x) < 500, random_state=42)
    ).copy()
    print("\nStratified Distribution:")
    print(df_sampled['Score_Tier'].value_counts().sort_index())
    if 'Score_Tier' in df_sampled.columns:
        df_sampled = df_sampled.drop(columns=['Score_Tier'])
    for col in ['rpg_url', 'rpgsystem', 'rpggenre', 'rpgsetting', 'rpgcategory']:
        df_sampled[col] = pd.Series(dtype='object')
    df_sampled = df_sampled.sample(frac=1, random_state=42).reset_index(drop=True)
    df_sampled.to_csv(SCRAPED_FILE, index=False)
    print(f"Created stratified sample of {len(df_sampled)} games and saved to '{SCRAPED_FILE}'.")


# ==========================================
# BLOCK 2: Scrape RPGGeek Features (curl_cffi browser impersonation)
# ==========================================
print("\n--- BLOCK 2: WEB SCRAPING FEATURES (HTML + curl_cffi) ---")

# curl_cffi impersonates real browser TLS fingerprints — this is what Cloudflare actually checks
scraper = curl_requests.Session(impersonate="chrome120")

human_headers = {
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Sec-Ch-Ua": '"Not_A Brand";v="8", "Chromium";v="120", "Google Chrome";v="120"',
    "Sec-Ch-Ua-Mobile": "?0",
    "Sec-Ch-Ua-Platform": '"Windows"',
    "Sec-Fetch-Dest": "document",
    "Sec-Fetch-Mode": "navigate",
    "Sec-Fetch-Site": "none",
    "Upgrade-Insecure-Requests": "1",
}

# Warm up: visit the homepage first so Cloudflare sees a natural session
print("Warming up session on BGG homepage...")
try:
    scraper.get("https://boardgamegeek.com/", headers=human_headers, timeout=15)
    time.sleep(random.uniform(3.0, 5.0))
    print("Session ready.")
except Exception as e:
    print(f"Warm-up warning (non-fatal): {e}")

missing_mask = df_sampled['rpg_url'].isna() & df_sampled['rpgsystem'].isna() & df_sampled['rpggenre'].isna()
missing_indices = df_sampled[missing_mask].index.tolist()

if not missing_indices:
    print("All rows have been processed! Moving to Block 3.")
else:
    print(f"Found {len(missing_indices)} games needing features scraped.")

    consecutive_403 = 0
    hard_exit = False

    for session_count, idx in enumerate(missing_indices):
        if hard_exit:
            break

        game_name = str(df_sampled.at[idx, 'Name'])
        print(f"[{session_count+1}/{len(missing_indices)}] {game_name[:25]:<25}", end=" ")

        try:
            # STEP A: Search page
            encoded_name = urllib.parse.quote(game_name)
            search_url = f"https://boardgamegeek.com/search/rpgitem?q={encoded_name}"
            res_search = scraper.get(search_url, headers=human_headers, timeout=15)
            time.sleep(random.uniform(3.0, 6.0))

            if res_search.status_code == 403:
                consecutive_403 += 1
                print(f"| 403 Blocked (Strike {consecutive_403}/3)")
                if consecutive_403 >= 3:
                    hard_exit = True
                continue

            consecutive_403 = 0
            soup_search = BeautifulSoup(res_search.content, 'html.parser')
            first_link = soup_search.find('a', href=re.compile(r'^/rpgitem/\d+/'))

            if not first_link:
                print("| No results found")
                df_sampled.at[idx, 'rpg_url'] = "NOT_FOUND"
            else:
                full_url = "https://boardgamegeek.com" + first_link['href']
                df_sampled.at[idx, 'rpg_url'] = full_url

                # STEP B: Game page
                res_page = scraper.get(full_url, headers=human_headers, timeout=15)
                time.sleep(random.uniform(3.0, 6.0))

                if res_page.status_code == 403:
                    consecutive_403 += 1
                    print(f"| 403 on game page (Strike {consecutive_403}/3)")
                    if consecutive_403 >= 3:
                        hard_exit = True
                    continue

                consecutive_403 = 0
                soup_page = BeautifulSoup(res_page.content, 'html.parser')

                found_data = False
                for script in soup_page.find_all('script'):
                    if script.string and 'GEEK.geekitemPreload' in script.string:
                        json_match = re.search(r'GEEK\.geekitemPreload\s*=\s*(\{.*?\});', script.string, re.DOTALL)
                        if json_match:
                            try:
                                data = json.loads(json_match.group(1))
                                links = data.get('item', {}).get('links', {})
                                for category in ['rpgsystem', 'rpggenre', 'rpgsetting', 'rpgcategory']:
                                    if category in links and isinstance(links[category], list):
                                        names = [item.get('name') for item in links[category] if item.get('name')]
                                        if names:
                                            df_sampled.at[idx, category] = ", ".join(names)
                                            found_data = True
                            except json.JSONDecodeError:
                                pass
                        break

                if found_data:
                    print("| OK - features extracted")
                else:
                    print("| No categories found in page")

        except Exception as e:
            print(f"| Error: {e}")
            time.sleep(5)

        if (session_count + 1) % 10 == 0:
            df_sampled.to_csv(SCRAPED_FILE, index=False)
            print(f"      Auto-saved at {session_count+1} items.")

    df_sampled.to_csv(SCRAPED_FILE, index=False)

    if hard_exit:
        print("\nSCRAPER STOPPED: 3 consecutive 403s.")
        print("TIP: Wait 10-15 minutes (or switch networks) then re-run — your progress is saved.")
        sys.exit("Terminated due to 403 limits.")

    print(f"\nBlock 2 complete. Saved to '{SCRAPED_FILE}'.")


# ==========================================
# BLOCK 3: One-Hot Encode All Features
# ==========================================
print("\n--- BLOCK 3: ONE-HOT ENCODING (MACHINE LEARNING PREP) ---")
df_ml = pd.read_csv(SCRAPED_FILE)

cols_to_encode = ['rpgsystem', 'rpggenre', 'rpgsetting', 'rpgcategory']

for col in cols_to_encode:
    df_ml[col] = df_ml[col].fillna('')

def one_hot_encode_column(df, column_name):
    s = df[column_name].str.split(', ').apply(lambda x: [item.strip() for item in x if item.strip()])
    if s.apply(len).sum() == 0:
        return pd.DataFrame(index=df.index)
    ohe_df = s.explode().str.get_dummies().groupby(level=0).sum()
    prefix = column_name.replace('rpg', '') + "_"
    ohe_df.columns = [prefix + col for col in ohe_df.columns]
    return ohe_df

print("Generating binary columns...")
ohe_dataframes = [df_ml]

for col in cols_to_encode:
    print(f"Encoding {col}...")
    ohe_df = one_hot_encode_column(df_ml, col)
    if not ohe_df.empty:
        ohe_dataframes.append(ohe_df)

df_final = pd.concat(ohe_dataframes, axis=1)
df_final = df_final.drop(columns=cols_to_encode + ['rpg_url'])
df_final.to_csv(FINAL_OHE_FILE, index=False)

print(f"\nPIPELINE COMPLETE!")
print(f"Original shape: {df_sampled.shape}")
print(f"Final OHE shape: {df_final.shape}")
print(f"Machine Learning ready dataset saved to '{FINAL_OHE_FILE}'.")


Starting the Autonomous TTRPG Feature Engineering Pipeline (HTML SCRAPE + curl_cffi)...

--- BLOCK 1: STRATIFIED SAMPLING DATA ---
Loaded main dataset with 10000 total games.
Resuming from existing 'scraped_rpg_features_part3.csv' with 1250 games.

--- BLOCK 2: WEB SCRAPING FEATURES (HTML + curl_cffi) ---
Warming up session on BGG homepage...
Session ready.
Found 1078 games needing features scraped.
[1/1078] Rüs                       | OK - features extracted
[2/1078] The Black Company         | OK - features extracted
[3/1078] Mutazoids                 | OK - features extracted
[4/1078] A Challenge of Arm's      | OK - features extracted
[5/1078] Ninth World Guidebook     | OK - features extracted
[6/1078] Ballots & Bullets         | OK - features extracted
[7/1078] Blade of the Young Samura | OK - features extracted
[8/1078] Book of Vile Darkness     | OK - features extracted
[9/1078] Don't F*** the Priest     | OK - features extracted
[10/1078] College of the Maestro    | OK - featu